# Patrón Estructural: Flyweight

## Introducción
El patrón Flyweight permite reducir el uso de memoria compartiendo la mayor cantidad de datos posible entre objetos similares. Es útil cuando hay muchos objetos que comparten información.

## Objetivos
- Comprender cómo optimizar el uso de memoria con objetos compartidos.
- Identificar cuándo es útil el patrón Flyweight.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Editor de texto**
En un editor de texto, cada carácter podría ser un objeto. El patrón Flyweight permite compartir la representación de los caracteres, ahorrando memoria.

**¿Dónde se usa en proyectos reales?**
En editores de texto, juegos (por ejemplo, árboles en un bosque), sistemas de gráficos, etc.

## Sin patrón Flyweight (forma errónea)
Cada objeto carácter almacena toda la información, lo que consume mucha memoria.

In [1]:
class Caracter:
    def __init__(self, simbolo, fuente):
        self.simbolo = simbolo
        self.fuente = fuente

texto = 'aca'
caracteres = [Caracter(s, 'Arial') for s in texto]
print(caracteres)

[<__main__.Caracter object at 0x10a87ba10>, <__main__.Caracter object at 0x10a5bb9d0>, <__main__.Caracter object at 0x10a5bbb10>]


## Con patrón Flyweight (forma correcta)
Se comparte la información común entre los objetos.

In [2]:
class FlyweightCaracter:
    _flyweights = {}
    def __new__(cls, simbolo, fuente):
        key = (simbolo, fuente)
        if key not in cls._flyweights:
            cls._flyweights[key] = super().__new__(cls)
        return cls._flyweights[key]
    def __init__(self, simbolo, fuente):
        self.simbolo = simbolo
        self.fuente = fuente

texto = 'aac'
caracteres = [FlyweightCaracter(s, 'Arial') for s in texto]
print(caracteres)

[<__main__.FlyweightCaracter object at 0x10a87be00>, <__main__.FlyweightCaracter object at 0x10a87be00>, <__main__.FlyweightCaracter object at 0x10a5bbc50>]


## UML del patrón Flyweight
```plantuml
@startuml
class FlyweightCaracter {
    + simbolo
    + fuente
}
FlyweightCaracter <.. Cliente
@enduml
```

## Otro ejemplo de la vida real: Renderizado de un bosque en un videojuego
**Contexto:** un motor de videojuegos necesita dibujar miles de árboles en un mapa. Cada árbol tiene una posición única (x, y), pero el modelo 3D y la textura son idénticos para todos los árboles del mismo tipo (ej. "pino"). Si cada instancia de árbol guarda su propia copia del modelo 3D y la textura (datos pesados), la memoria se dispara con miles de árboles.

### Sin patrón (forma errónea)
Cada árbol almacena su propia copia del modelo 3D y la textura, aunque sean idénticos entre árboles del mismo tipo.

In [3]:
class Arbol:
    def __init__(self, x, y, modelo_3d, textura):
        self.x = x
        self.y = y
        self.modelo_3d = modelo_3d  # dato pesado, duplicado en cada instancia
        self.textura = textura      # dato pesado, duplicado en cada instancia

bosque = [Arbol(x, y, 'modelo_pino.obj', 'textura_pino.png') for x, y in [(1, 2), (5, 7), (9, 1)]]
print(f'{len(bosque)} árboles, cada uno con su propia copia de modelo y textura')

3 árboles, cada uno con su propia copia de modelo y textura


### Con patrón (forma correcta)
El estado intrínseco (modelo 3D, textura) se comparte en un `TipoArbol` flyweight; cada `Arbol` solo guarda su estado extrínseco (posición) y una referencia al flyweight.

In [4]:
class TipoArbol:
    _tipos = {}
    def __new__(cls, modelo_3d, textura):
        key = (modelo_3d, textura)
        if key not in cls._tipos:
            cls._tipos[key] = super().__new__(cls)
        return cls._tipos[key]
    def __init__(self, modelo_3d, textura):
        self.modelo_3d = modelo_3d
        self.textura = textura
    def dibujar(self, x, y):
        print(f'Dibujando {self.modelo_3d} en ({x}, {y})')


class Arbol:
    def __init__(self, x, y, tipo: TipoArbol):
        self.x = x
        self.y = y
        self.tipo = tipo  # referencia compartida, no una copia
    def dibujar(self):
        self.tipo.dibujar(self.x, self.y)


tipo_pino = TipoArbol('modelo_pino.obj', 'textura_pino.png')
bosque = [Arbol(x, y, tipo_pino) for x, y in [(1, 2), (5, 7), (9, 1)]]

for arbol in bosque:
    arbol.dibujar()

print(bosque[0].tipo is bosque[1].tipo)

Dibujando modelo_pino.obj en (1, 2)
Dibujando modelo_pino.obj en (5, 7)
Dibujando modelo_pino.obj en (9, 1)
True


### UML del ejemplo del bosque
```plantuml
@startuml
class TipoArbol {
    - _tipos: dict
    + modelo_3d
    + textura
    + dibujar(x, y)
}
class Arbol {
    + x
    + y
    + tipo: TipoArbol
    + dibujar()
}
Arbol --> TipoArbol : comparte
@enduml
```

### ¿Dónde más se usa Flyweight?
- **Videojuegos con muchos objetos repetidos:** exactamente este ejemplo — árboles, balas, enemigos idénticos que solo difieren en posición/estado.
- **Editores de texto:** compartir el glifo/fuente de cada carácter en vez de duplicarlo por cada letra escrita (el ejemplo con el que abre este notebook).
- **Renderizado de mapas:** miles de marcadores/pines en un mapa comparten el mismo ícono y solo varían sus coordenadas.
- **Pools de objetos inmutables:** el internado de strings en Python (`sys.intern`) o el cacheo de enteros pequeños reutiliza el mismo objeto en memoria.
- **Sistemas de partículas:** miles de partículas (chispas, lluvia) comparten la textura y el comportamiento base, y solo varían posición y velocidad.

**Ejercicio de reflexión:** si el bosque necesitara árboles con distinta altura además de posición, ¿la altura debería ir en `TipoArbol` (estado intrínseco/compartido) o en `Arbol` (estado extrínseco/único)? Justifica tu respuesta.

## Actividad
Crea un sistema que use Flyweight para representar piezas de ajedrez en un tablero, compartiendo la información común.

---
## Explicación de conceptos clave
- **Ahorro de memoria:** Compartir datos comunes entre muchos objetos.
- **Separación de estado:** El estado compartido se almacena en el flyweight, el estado único en el cliente.
- **Aplicación en la vida real:** Útil en editores de texto, juegos y sistemas gráficos.

## Conclusión
El patrón Flyweight es ideal para optimizar el uso de memoria cuando hay muchos objetos similares. Es común en editores de texto, juegos y aplicaciones gráficas.